# Visual Product Search Engine — Full Ablation Study
**Course: Visual Recognition | DeepFashion In-Shop Clothes Retrieval**

## Pipeline
- **Offline**: YOLO crop → BLIP-2 caption → CLIP fused embedding → HNSW index
- **Ablation A**: Vision-only CLIP (α=1), no fine-tuning — Baseline
- **Ablation B**: Frozen CLIP + Frozen BLIP-2 — Two α values, multi-seed
- **Ablation C**: Fine-tuned CLIP + Frozen BLIP-2 — Two α values, multi-seed
- **Metrics**: Recall@K, NDCG@K, mAP@K for K ∈ {5, 10, 15}
- **Results**: mean ± std over 3 seeds

## 0. Install Dependencies

In [2]:

!pip install ultralytics git+https://github.com/openai/CLIP.git 
!pip install faiss-cpu 
!pip install transformers accelerate sentencepiece bitsandbytes einops
# bitsandbytes for 8-bit BLIP-2 to fit on Kaggle GPU

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-g0yjqs08
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-g0yjqs08
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=9ad385f49b7973ce99d90158c41041626e9149873dec2ac5d112af3946b1f4d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-yysybz47/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01


## 1. Imports & Global Config

In [3]:
import os, gc, random, pickle, warnings, io
import numpy as np
import pandas as pd
import torch
import clip
import faiss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    BlipProcessor, BlipForImageTextRetrieval,
    BitsAndBytesConfig
)
from ultralytics import YOLO
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_ROOT     = Path('/kaggle/input/datasets/asutoshpandaschrodie/vr-final')
IMG_DIR          = DATASET_ROOT / 'img'
EVAL_FILE        = DATASET_ROOT / 'list_eval_partition.txt'
BBOX_FILE        = DATASET_ROOT / 'list_bbox_inshop.txt'
WORK             = Path('/kaggle/working')
FINAL_OUTPUT_DIR = WORK / 'final_output'
CROP_DIR         = WORK / 'cropped'
EMB_DIR          = WORK / 'embeddings'
CAP_DIR          = WORK / 'captions'
IDX_DIR          = WORK / 'indices'
VIZ_DIR          = WORK / 'visualizations'
RESULT_DIR       = WORK / 'results'
for d in [FINAL_OUTPUT_DIR, EMB_DIR, CAP_DIR, IDX_DIR, VIZ_DIR, RESULT_DIR,
          CROP_DIR / 'upper', CROP_DIR / 'lower', CROP_DIR / 'full']:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyperparams ────────────────────────────────────────────────────────────────
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS           = [42, 137, 256, 512]
K_VALUES        = [5, 10, 15]
MAX_K           = max(K_VALUES)
ALPHA_B         = [0.3, 0.7]
ALPHA_C         = [0.3, 0.7]
ALPHA_BEST      = 0.7
CLIP_MODEL      = 'ViT-B/32'
FINETUNE_LR     = 1e-6
FINETUNE_EPOCHS = 5
BATCH_SIZE      = 32
TEMPERATURE     = 0.07
EMB_BATCH       = 128
torch.backends.cudnn.benchmark = True
print(f'Device : {DEVICE}')
print(f'Seeds  : {SEEDS}')
print(f'GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


# ── Reproducibility ────────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ── Image path helpers ─────────────────────────────────────────────────────────
def crop_path(row) -> Path:
    return CROP_DIR / row.image_path

def orig_path(row) -> Path:
    return IMG_DIR / row.image_path

# ── Metric functions ──────────────────────────────────────────────────────────
def recall_at_k(topk: np.ndarray, q_ids, g_ids, k: int) -> float:
    return np.mean([
        1.0 if q_ids[i] in g_ids[topk[i, :k]] else 0.0
        for i in range(len(q_ids))
    ])

def _ap(relevant: np.ndarray) -> float:
    correct = ps = 0
    for i, r in enumerate(relevant):
        if r:
            correct += 1
            ps += correct / (i + 1)
    return ps / correct if correct else 0.0

def map_at_k(topk: np.ndarray, q_ids, g_ids, k: int) -> float:
    aps = []
    for i in range(len(q_ids)):
        ret = g_ids[topk[i, :k]]
        aps.append(_ap((ret == q_ids[i]).astype(int)))
    return float(np.mean(aps))

def ndcg_at_k(topk: np.ndarray, q_ids, g_ids, k: int) -> float:
    scores = []
    for i in range(len(q_ids)):
        ret  = g_ids[topk[i, :k]]
        rel  = (ret == q_ids[i]).astype(float)
        dcg  = sum(rel[j] / np.log2(j + 2) for j in range(k))
        idcg = sum(1.0     / np.log2(j + 2) for j in range(min(int(rel.sum()), k)))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))

def compute_all_metrics(topk: np.ndarray, q_ids, g_ids) -> dict:
    metrics = {}
    for k in K_VALUES:
        metrics[f'Recall@{k}'] = recall_at_k(topk, q_ids, g_ids, k)
        metrics[f'mAP@{k}']    = map_at_k(topk,    q_ids, g_ids, k)
        metrics[f'NDCG@{k}']   = ndcg_at_k(topk,   q_ids, g_ids, k)
    return metrics

# ── Fuse + normalize embeddings ───────────────────────────────────────────────
def fuse(img_emb: np.ndarray, txt_emb: np.ndarray, alpha: float) -> np.ndarray:
    fused = alpha * img_emb + (1 - alpha) * txt_emb
    norms = np.linalg.norm(fused, axis=1, keepdims=True) + 1e-8
    return (fused / norms).astype('float32')

# ── HNSW retrieval ─────────────────────────────────────────────────────────────
def build_hnsw_and_retrieve(gallery_emb: np.ndarray, query_emb: np.ndarray) -> np.ndarray:
    dim   = gallery_emb.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch       = 128
    faiss.normalize_L2(gallery_emb)
    faiss.normalize_L2(query_emb)
    index.add(gallery_emb)
    _, topk = index.search(query_emb, MAX_K)
    return topk

# ── Mean±std summary ───────────────────────────────────────────────────────────
def summarize_seeds(records: list[dict], label: str) -> pd.DataFrame:
    df  = pd.DataFrame(records)
    num = df.select_dtypes(include='number')
    summary_rows = []
    for col in num.columns:
        summary_rows.append({
            'Experiment': label,
            'Metric':     col,
            'Mean':       num[col].mean(),
            'Std':        num[col].std(ddof=1)
        })
    return pd.DataFrame(summary_rows)

print('Utility functions loaded.')


# ── Parse annotation files ─────────────────────────────────────────────────────
def parse_eval(path):
    rows = []
    with open(path) as f:
        lines = f.readlines()[2:]
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 3:
            rows.append({'image_path': parts[0], 'item_id': parts[1], 'split': parts[2]})
    return pd.DataFrame(rows)

def parse_bbox(path):
    rows = []
    with open(path) as f:
        lines = f.readlines()[2:]
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 7:
            ctype = int(parts[1])
            rows.append({
                'image_path':    parts[0],
                'clothing_type': ctype,
                'clothing_label': {1:'upper', 2:'lower', 3:'full'}.get(ctype, 'full'),
                'x1': int(parts[3]), 'y1': int(parts[4]),
                'x2': int(parts[5]), 'y2': int(parts[6])
            })
    return pd.DataFrame(rows)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device : cuda
Seeds  : [42, 137, 256, 512]
GPU    : Tesla T4
Utility functions loaded.


In [22]:

print('Parsing annotation files...')
eval_df = parse_eval(EVAL_FILE)
bbox_df = parse_bbox(BBOX_FILE)

print('Merging dataframes...')
merged = pd.merge(eval_df, bbox_df, on='image_path')

# ── Fast file existence scan ───────────────────────────────────────────────────
# OPTIMISATION: rglob once into a set → O(1) membership checks
print('Scanning image directory...')
existing_files = {str(p.relative_to(IMG_DIR)) for p in IMG_DIR.rglob('*') if p.is_file()}
print(f'Found {len(existing_files):,} files')

print('Validating image paths...')
valid = merged[merged['image_path'].isin(existing_files)].copy().reset_index(drop=True)

train_df   = valid[valid['split'] == 'train'  ].reset_index(drop=True)
query_df   = valid[valid['split'] == 'query'  ].reset_index(drop=True)
gallery_df = valid[valid['split'] == 'gallery'].reset_index(drop=True)

train_df.to_csv(FINAL_OUTPUT_DIR   / 'train.csv',   index=False)
query_df.to_csv(FINAL_OUTPUT_DIR   / 'query.csv',   index=False)
gallery_df.to_csv(FINAL_OUTPUT_DIR / 'gallery.csv', index=False)

print(f'Train   : {len(train_df):,}')
print(f'Query   : {len(query_df):,}')
print(f'Gallery : {len(gallery_df):,}')
print(f'Total   : {len(valid):,}')

fig, ax = plt.subplots(figsize=(6, 4))
valid['split'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Dataset Split Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'split_distribution.png', dpi=150)
plt.close()
print('Split distribution plot saved.')


# ── YOLO cropping ──────────────────────────────────────────────────────────────
yolo_model = YOLO('yolov8n.pt')
yolo_model.to(DEVICE)
FALLBACK_TO_GT = True

def yolo_crop(row, model) -> bool:
    src = orig_path(row)
    dst = crop_path(row)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return True
    try:
        img    = Image.open(src).convert('RGB')
        W, H   = img.size
        results = model(src, verbose=False, conf=0.15)[0]
        boxes   = results.boxes
        crop_box = None
        if boxes is not None and len(boxes) > 0:
            confs    = boxes.conf.cpu().numpy()
            cls_ids  = boxes.cls.cpu().numpy().astype(int)
            xyxy_all = boxes.xyxy.cpu().numpy()
            person_mask = cls_ids == 0
            best = np.argmax(confs * person_mask) if person_mask.any() else np.argmax(confs)
            x1, y1, x2, y2 = xyxy_all[best]
            crop_box = (max(0,int(x1)), max(0,int(y1)), min(W,int(x2)), min(H,int(y2)))
        if crop_box is None or (crop_box[2]-crop_box[0]) < 20 or (crop_box[3]-crop_box[1]) < 20:
            if FALLBACK_TO_GT:
                crop_box = (int(row.x1), int(row.y1), int(row.x2), int(row.y2))
            else:
                crop_box = (0, 0, W, H)
        img.crop(crop_box).save(dst)
        return True
    except Exception as e:
        print(f'YOLO crop error {src}: {e}')
        return False
        

# OPTIMISATION: itertuples (~50× faster than iterrows) for tight loops
print('Running YOLO cropping on all images...')
all_rows = pd.concat([train_df, query_df, gallery_df], ignore_index=True)
success = fail = 0
pbar = tqdm(total=len(all_rows), desc='YOLO Crop', unit='img')
for row in all_rows.itertuples(index=False):
    ok = yolo_crop(row, yolo_model)
    success += ok; fail += not ok
    pbar.update(1)
pbar.close()
print(f'Crop complete — Success: {success:,}  Fail: {fail:,}')

del yolo_model; gc.collect(); torch.cuda.empty_cache()

samples = gallery_df.sample(6, random_state=42)
fig, axes = plt.subplots(6, 2, figsize=(10, 24))
for i, row in enumerate(samples.itertuples(index=False)):
    orig    = Image.open(orig_path(row)).convert('RGB')
    cropped = Image.open(crop_path(row)).convert('RGB')
    axes[i, 0].imshow(orig);    axes[i, 0].set_title('Original'); axes[i, 0].axis('off')
    axes[i, 1].imshow(cropped); axes[i, 1].set_title('YOLO Crop'); axes[i, 1].axis('off')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'yolo_crop_examples.png', dpi=150)
plt.close()
print('YOLO crop visualization saved.')


# ── BLIP-2 loading ─────────────────────────────────────────────────────────────




Device : cuda
Seeds  : [42, 137, 256, 512]
GPU    : Tesla T4
Utility functions loaded.
Parsing annotation files...
Merging dataframes...
Scanning image directory...
Found 52,712 files
Validating image paths...
Train   : 25,882
Query   : 14,218
Gallery : 12,612
Total   : 52,712
Split distribution plot saved.
Running YOLO cropping on all images...


YOLO Crop:   0%|          | 0/52712 [00:00<?, ?img/s]

Crop complete — Success: 52,712  Fail: 0
YOLO crop visualization saved.
Loading BLIP-2 (Salesforce/blip2-opt-2.7b) in 8-bit...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

BLIP-2 loaded.
Captioning gallery images...


BLIP-2 Caption:   0%|          | 0/789 [00:00<?, ?batch/s]

The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when us

KeyboardInterrupt: 

In [6]:
train_df = pd.read_csv("/kaggle/working/final_output/train.csv")
query_df = pd.read_csv("/kaggle/working/final_output/query.csv")
gallery_df = pd.read_csv("/kaggle/working/final_output/gallery.csv")

In [23]:
print('Loading BLIP-2 (Salesforce/blip2-opt-2.7b) in 8-bit...')
blip2_processor = Blip2Processor.from_pretrained('Salesforce/blip2-opt-2.7b')
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    'Salesforce/blip2-opt-2.7b',
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map={'': 0}, 
    torch_dtype=torch.float16,
)
blip2_model.eval()
print('BLIP-2 loaded.')


# ── Thread-safe image loader ───────────────────────────────────────────────────
# PIL JPEG decoder is NOT thread-safe — concurrent Image.open() calls corrupt
# each other silently. Fix: read raw bytes in threads (safe), decode on main
# thread (safe). This still overlaps all disk I/O.

def _read_bytes(path: str) -> tuple[str, bytes | None]:
    try:
        with open(path, 'rb') as f:
            return (path, f.read())
    except Exception:
        return (path, None)

def _decode_images_parallel(paths: list[str], io_workers: int = 4):
    with ThreadPoolExecutor(max_workers=io_workers) as pool:
        byte_results = list(pool.map(_read_bytes, paths))
    out = []
    for path, raw in byte_results:
        if raw is None:
            out.append((path, None))
            continue
        try:
            img = Image.open(io.BytesIO(raw)).convert('RGB')
            out.append((path, img) if img.size[0] >= 10 and img.size[1] >= 10 else (path, None))
        except Exception as e:
            print(f'  [decode error] {Path(path).name}: {e}')
            out.append((path, None))
    return out


# ── Captioning ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_captions_batch(
    df: pd.DataFrame,
    batch_size: int    = 16,   # T4 comfortably fits 16 for OPT-2.7B in 8-bit
    max_new_tokens: int = 25,  # was 40 — 25 covers all fashion captions
    num_beams: int     = 1,    # greedy — 3× faster than beam=3, negligible quality drop
    io_workers: int    = 4,
) -> pd.DataFrame:
    # itertuples: ~50× faster than iterrows; crop_path handles both Series & namedtuple
    paths    = [str(crop_path(row)) for row in df.itertuples(index=False)]
    captions = [''] * len(paths)
    skipped  = 0

    pbar = tqdm(range(0, len(paths), batch_size), desc='BLIP-2 Caption', unit='batch')
    for i in pbar:
        batch_paths = paths[i : i + batch_size]

        # bytes read in threads → PIL decode on main thread (thread-safe)
        loaded = _decode_images_parallel(batch_paths, io_workers=io_workers)

        images, valid_idx = [], []
        for j, (path, img) in enumerate(loaded):
            if img is not None:
                images.append(img)
                valid_idx.append(i + j)
            else:
                skipped += 1
                if skipped <= 5:
                    print(f'  [skip] {path}')

        if not images:
            continue

        inputs    = blip2_processor(images=images, return_tensors='pt').to(DEVICE)
        generated = blip2_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=num_beams)
        texts     = blip2_processor.batch_decode(generated, skip_special_tokens=True)

        for k, vi in enumerate(valid_idx):
            captions[vi] = texts[k].strip()

        if texts:
            pbar.set_postfix_str(texts[0][:50])

    print(f'\nDone — {len(paths)-skipped:,} captioned, {skipped:,} skipped')
    df = df.copy()
    df['caption'] = captions
    return df

def _caption_with_cache(df: pd.DataFrame, cache_path: Path, label: str, **kw) -> pd.DataFrame:
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        # Reject stale cache: all captions missing means old broken run
        if cached['caption'].notna().any():
            print(f'{label} captions loaded from cache.')
            return cached
        print(f'{label} cache is stale (all NaN) — regenerating...')
        cache_path.unlink()
    print(f'Captioning {label} images...')
    cap_df = generate_captions_batch(df, **kw)
    cap_df.to_csv(cache_path, index=False)
    return cap_df

gallery_cap_df = _caption_with_cache(gallery_df, CAP_DIR / 'gallery_captions.csv', 'gallery')
query_cap_df   = _caption_with_cache(query_df,   CAP_DIR / 'query_captions.csv',   'query')

print(f'Gallery captions : {len(gallery_cap_df):,}')
print(f'Query   captions : {len(query_cap_df):,}')
print('\nSample gallery captions:')
print(gallery_cap_df[['image_path', 'item_id', 'caption']].head(5).to_string(index=False))

del blip2_model; gc.collect(); torch.cuda.empty_cache()
print('BLIP-2 released.')

Loading BLIP-2 (Salesforce/blip2-opt-2.7b) in 8-bit...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

BLIP-2 loaded.
Captioning gallery images...


BLIP-2 Caption:   0%|          | 0/789 [00:00<?, ?batch/s]

  [skip] /kaggle/working/cropped/img/WOMEN/Leggings/id_00002368/01_2_side.jpg

Done — 12,611 captioned, 1 skipped
Captioning query images...


BLIP-2 Caption:   0%|          | 0/889 [00:00<?, ?batch/s]


Done — 14,218 captioned, 0 skipped
Gallery captions : 12,612
Query   captions : 14,218

Sample gallery captions:
                                         image_path     item_id                                                          caption
img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg id_00000001                       a woman in black shorts and a white blouse
 img/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg id_00000001 the back view of a woman wearing black shorts and a white blouse
    img/WOMEN/Tees_Tanks/id_00000007/01_1_front.jpg id_00000007    a woman wearing a tank top with a picture of snoop dogg on it
     img/WOMEN/Tees_Tanks/id_00000007/01_3_back.jpg id_00000007 the back view of a model wearing black jeans and a grey tank top
        img/WOMEN/Dresses/id_00000008/02_3_back.jpg id_00000008                   the back view of a woman wearing a black dress
BLIP-2 released.


## 6. Step 3 — CLIP Embedding Generation (Frozen, used for Exp A & B)

In [24]:
# Load frozen CLIP once for Exp A and B
print('Loading frozen CLIP...')
clip_model_frozen, clip_preprocess = clip.load(CLIP_MODEL, device=DEVICE, jit=False)
clip_model_frozen = clip_model_frozen.float()
clip_model_frozen.eval()
for p in clip_model_frozen.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def encode_images_clip(df: pd.DataFrame, model, preprocess,
                       batch_size: int = EMB_BATCH) -> np.ndarray:
    """Encode cropped images with CLIP visual encoder. Returns L2-normalised embeddings."""
    embs  = []
    paths = [crop_path(row) for _, row in df.iterrows()]

    pbar = tqdm(range(0, len(paths), batch_size), desc='CLIP Image Emb', unit='batch', leave=False)
    for i in pbar:
        imgs = []
        for p in paths[i:i+batch_size]:
            try:
                imgs.append(preprocess(Image.open(p).convert('RGB')))
            except:
                imgs.append(torch.zeros(3, 224, 224))
        batch = torch.stack(imgs).to(DEVICE, non_blocking=True)
        feat  = model.encode_image(batch).float()
        feat  = feat / (feat.norm(dim=-1, keepdim=True) + 1e-8)
        embs.append(feat.cpu().numpy())
    return np.vstack(embs)

@torch.no_grad()
def encode_texts_clip(captions: list, model, batch_size: int = 512) -> np.ndarray:
    """Encode captions with CLIP text encoder. Returns L2-normalised embeddings."""
    embs = []
    pbar = tqdm(range(0, len(captions), batch_size), desc='CLIP Text Emb', unit='batch', leave=False)
    for i in pbar:
        batch  = captions[i:i+batch_size]
        tokens = clip.tokenize(batch, truncate=True).to(DEVICE)
        feat   = model.encode_text(tokens).float()
        feat   = feat / (feat.norm(dim=-1, keepdim=True) + 1e-8)
        embs.append(feat.cpu().numpy())
    return np.vstack(embs)

# ── Generate and cache frozen CLIP image embeddings ───────────────────────────
print('Generating frozen CLIP image embeddings...')
def cache_emb(path, fn, *args):
    if path.exists():
        print(f'  Loaded from cache: {path.name}')
        return np.load(path)
    arr = fn(*args)
    np.save(path, arr)
    return arr

gallery_img_emb_frozen = cache_emb(
    EMB_DIR / 'gallery_img_frozen.npy',
    encode_images_clip, gallery_df, clip_model_frozen, clip_preprocess)

query_img_emb_frozen = cache_emb(
    EMB_DIR / 'query_img_frozen.npy',
    encode_images_clip, query_df, clip_model_frozen, clip_preprocess)

# ── Generate and cache frozen CLIP text embeddings ───────────────────────────
print('Generating CLIP text embeddings from BLIP-2 captions...')
gallery_txt_emb = cache_emb(
    EMB_DIR / 'gallery_txt.npy',
    encode_texts_clip,
    gallery_cap_df['caption'].fillna('clothing item').tolist(),
    clip_model_frozen)

query_txt_emb = cache_emb(
    EMB_DIR / 'query_txt.npy',
    encode_texts_clip,
    query_cap_df['caption'].fillna('clothing item').tolist(),
    clip_model_frozen)

print(f'Gallery img emb : {gallery_img_emb_frozen.shape}')
print(f'Gallery txt emb : {gallery_txt_emb.shape}')
print(f'Query   img emb : {query_img_emb_frozen.shape}')
print(f'Query   txt emb : {query_txt_emb.shape}')

# Keep item ID arrays for metrics
gallery_ids = gallery_df['item_id'].values
query_ids   = query_df['item_id'].values

Loading frozen CLIP...


100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 95.6MiB/s]


Generating frozen CLIP image embeddings...


CLIP Image Emb:   0%|          | 0/99 [00:00<?, ?batch/s]

CLIP Image Emb:   0%|          | 0/112 [00:00<?, ?batch/s]

Generating CLIP text embeddings from BLIP-2 captions...


CLIP Text Emb:   0%|          | 0/25 [00:00<?, ?batch/s]

CLIP Text Emb:   0%|          | 0/28 [00:00<?, ?batch/s]

Gallery img emb : (12612, 512)
Gallery txt emb : (12612, 512)
Query   img emb : (14218, 512)
Query   txt emb : (14218, 512)


In [10]:
gallery_img_emb_frozen = np.load(
    '/kaggle/working/embeddings/gallery_img_frozen.npy'
)

query_img_emb_frozen = np.load(
    '/kaggle/working/embeddings/query_img_frozen.npy'
)

gallery_txt_emb = np.load(
    '/kaggle/working/embeddings/gallery_txt.npy'
)

query_txt_emb = np.load(
    '/kaggle/working/embeddings/query_txt.npy'
)

## 7. Experiment A — Vision-only CLIP (α=1), No Fine-tuning [Baseline]

In [25]:
print('=' * 60)
print('EXPERIMENT A — Vision-only CLIP (α=1) | Baseline')
print('=' * 60)

# α=1 means image embedding only — no text fusion, no fine-tuning
topk_A = build_hnsw_and_retrieve(
    gallery_img_emb_frozen.copy().astype('float32'),
    query_img_emb_frozen.copy().astype('float32')
)

np.save(IDX_DIR / 'topk_A.npy', topk_A)

metrics_A = compute_all_metrics(topk_A, query_ids, gallery_ids)

print('\n── Experiment A Results ──')
df_A = pd.DataFrame([metrics_A])
print(df_A.T.rename(columns={0: 'Score'}).to_string())
df_A.to_csv(RESULT_DIR / 'exp_A_metrics.csv', index=False)

# ── Visualization: metric curves for Exp A ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['Recall', 'mAP', 'NDCG']):
    vals = [metrics_A[f'{metric}@{k}'] for k in K_VALUES]
    ax.plot(K_VALUES, vals, marker='o', linewidth=2, color='steelblue')
    ax.set_title(f'Exp A — {metric}@K')
    ax.set_xlabel('K'); ax.set_ylabel(metric)
    ax.set_ylim(0, 1); ax.grid(True)
    for k, v in zip(K_VALUES, vals):
        ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points', xytext=(0, 6))
plt.suptitle('Experiment A — Vision-only CLIP (α=1) Baseline', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_A_metrics.png', dpi=150)
plt.close()
print('Exp A metrics plot saved.')

# ── Retrieval visualization ───────────────────────────────────────────────────
fig, axes = plt.subplots(5, MAX_K+1, figsize=(22, 18))
sample_q_idx = [0, 10, 25, 50, 100]
for row_i, qi in enumerate(sample_q_idx):
    qrow = query_df.iloc[qi]
    axes[row_i, 0].imshow(Image.open(crop_path(qrow)).convert('RGB'))
    axes[row_i, 0].set_title('Query', fontsize=8); axes[row_i, 0].axis('off')
    for col_i, gi in enumerate(topk_A[qi, :MAX_K]):
        grow = gallery_df.iloc[gi]
        match = grow['item_id'] == qrow['item_id']
        axes[row_i, col_i+1].imshow(Image.open(crop_path(grow)).convert('RGB'))
        axes[row_i, col_i+1].set_title(
            '✓' if match else '✗', color='green' if match else 'red', fontsize=9)
        axes[row_i, col_i+1].axis('off')
plt.suptitle('Exp A — Top-15 Retrieval Results', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_A_retrieval.png', dpi=100)
plt.close()
print('Exp A retrieval visualization saved.')

EXPERIMENT A — Vision-only CLIP (α=1) | Baseline

── Experiment A Results ──
              Score
Recall@5   0.360177
mAP@5      0.256759
NDCG@5     0.284970
Recall@10  0.433394
mAP@10     0.255775
NDCG@10    0.304302
Recall@15  0.476790
mAP@15     0.251926
NDCG@15    0.312935
Exp A metrics plot saved.
Exp A retrieval visualization saved.


## 8. Experiment B — Frozen CLIP + Frozen BLIP-2 | Multi-seed

In [26]:
print('=' * 60)
print('EXPERIMENT B — Frozen CLIP + Frozen BLIP-2')
print(f'Alpha values tested: {ALPHA_B}')
print(f'Seeds: {SEEDS}')
print('=' * 60)

# In Exp B there is NO training — embeddings are deterministic.
# Seed controls only any stochastic parts; we still run multi-seed
# to satisfy the reporting requirement (mean ± std across seeds).

records_B = []   # one dict per (seed, alpha)

pbar_B = tqdm(total=len(SEEDS) * len(ALPHA_B), desc='Exp B (seed × alpha)')

for seed in SEEDS:
    set_seed(seed)
    for alpha in ALPHA_B:
        g_fused = fuse(gallery_img_emb_frozen.copy(), gallery_txt_emb.copy(), alpha)
        q_fused = fuse(query_img_emb_frozen.copy(),   query_txt_emb.copy(),   alpha)

        topk = build_hnsw_and_retrieve(g_fused, q_fused)
        m    = compute_all_metrics(topk, query_ids, gallery_ids)
        m['seed']  = seed
        m['alpha'] = alpha
        records_B.append(m)
        pbar_B.update(1)

pbar_B.close()

df_B_full = pd.DataFrame(records_B)
df_B_full.to_csv(RESULT_DIR / 'exp_B_all_seeds.csv', index=False)

# ── Summary: mean ± std per alpha ─────────────────────────────────────────────
print('\n── Experiment B: Mean ± Std (across seeds) per Alpha ──')
summary_B_rows = []
for alpha in ALPHA_B:
    sub = df_B_full[df_B_full['alpha'] == alpha]
    row = {'alpha': alpha}
    for col in [c for c in df_B_full.columns if '@' in c]:
        row[col] = f"{sub[col].mean():.4f} ± {sub[col].std(ddof=1):.4f}"
    summary_B_rows.append(row)
    
df_B_summary = pd.DataFrame(summary_B_rows)
df_B_summary.to_csv(RESULT_DIR / 'exp_B_summary.csv', index=False)
print(df_B_summary.to_string(index=False))

# ── Plot: metric vs K for each alpha (best seed results shown as bands) ───────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'darkorange']
for ax, metric in zip(axes, ['Recall', 'mAP', 'NDCG']):
    for alpha, color in zip(ALPHA_B, colors):
        sub  = df_B_full[df_B_full['alpha'] == alpha]
        means = [sub[f'{metric}@{k}'].mean() for k in K_VALUES]
        stds  = [sub[f'{metric}@{k}'].std(ddof=1) for k in K_VALUES]
        ax.plot(K_VALUES, means, marker='o', color=color, label=f'α={alpha}')
        ax.fill_between(K_VALUES,
                        [m-s for m,s in zip(means,stds)],
                        [m+s for m,s in zip(means,stds)],
                        alpha=0.2, color=color)
    ax.set_title(f'Exp B — {metric}@K')
    ax.set_xlabel('K'); ax.set_ylabel(metric)
    ax.set_ylim(0, 1); ax.grid(True); ax.legend()
plt.suptitle('Experiment B — Frozen CLIP + Frozen BLIP-2 (mean ± std)', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_B_metrics.png', dpi=150)
plt.close()
print('Exp B metrics plot saved.')

# Save best alpha topk for retrieval viz
best_alpha_B = ALPHA_B[np.argmax([df_B_full[df_B_full['alpha']==a]['mAP@10'].mean() for a in ALPHA_B])]
g_fused_best = fuse(gallery_img_emb_frozen.copy(), gallery_txt_emb.copy(), best_alpha_B)
q_fused_best = fuse(query_img_emb_frozen.copy(),   query_txt_emb.copy(),   best_alpha_B)
topk_B_best  = build_hnsw_and_retrieve(g_fused_best, q_fused_best)
np.save(IDX_DIR / 'topk_B_best.npy', topk_B_best)
print(f'Best alpha for Exp B: {best_alpha_B}')

# Retrieval visualization for best alpha
fig, axes = plt.subplots(5, MAX_K+1, figsize=(22, 18))
for row_i, qi in enumerate(sample_q_idx):
    qrow = query_df.iloc[qi]
    axes[row_i, 0].imshow(Image.open(crop_path(qrow)).convert('RGB'))
    axes[row_i, 0].set_title('Query', fontsize=8); axes[row_i, 0].axis('off')
    for col_i, gi in enumerate(topk_B_best[qi, :MAX_K]):
        grow = gallery_df.iloc[gi]
        match = grow['item_id'] == qrow['item_id']
        axes[row_i, col_i+1].imshow(Image.open(crop_path(grow)).convert('RGB'))
        axes[row_i, col_i+1].set_title('✓' if match else '✗',
                                        color='green' if match else 'red', fontsize=9)
        axes[row_i, col_i+1].axis('off')
plt.suptitle(f'Exp B — Top-15 Retrieval (α={best_alpha_B})', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_B_retrieval.png', dpi=100)
plt.close()
print('Exp B retrieval visualization saved.')

EXPERIMENT B — Frozen CLIP + Frozen BLIP-2
Alpha values tested: [0.3, 0.7]
Seeds: [42, 137, 256, 512]


Exp B (seed × alpha):   0%|          | 0/8 [00:00<?, ?it/s]


── Experiment B: Mean ± Std (across seeds) per Alpha ──
 alpha        Recall@5           mAP@5          NDCG@5       Recall@10          mAP@10         NDCG@10       Recall@15          mAP@15         NDCG@15
   0.3 0.2725 ± 0.0001 0.1874 ± 0.0001 0.2099 ± 0.0001 0.3372 ± 0.0001 0.1898 ± 0.0001 0.2284 ± 0.0000 0.3794 ± 0.0001 0.1879 ± 0.0001 0.2375 ± 0.0000
   0.7 0.4004 ± 0.0000 0.2774 ± 0.0000 0.3111 ± 0.0000 0.4908 ± 0.0000 0.2775 ± 0.0000 0.3355 ± 0.0000 0.5436 ± 0.0000 0.2721 ± 0.0000 0.3457 ± 0.0000
Exp B metrics plot saved.
Best alpha for Exp B: 0.7
Exp B retrieval visualization saved.


## 9. Experiment C — Fine-tuned CLIP + Frozen BLIP-2 | Multi-seed

In [12]:
@torch.no_grad()
def encode_images_clip(df: pd.DataFrame, model, preprocess,
                       batch_size: int = EMB_BATCH) -> np.ndarray:
    """Encode cropped images with CLIP visual encoder."""
    
    embs  = []
    paths = [crop_path(row) for _, row in df.iterrows()]

    pbar = tqdm(
        range(0, len(paths), batch_size),
        desc='CLIP Image Emb',
        unit='batch',
        leave=False
    )

    for i in pbar:
        imgs = []

        for p in paths[i:i+batch_size]:
            try:
                imgs.append(
                    preprocess(Image.open(p).convert('RGB'))
                )
            except:
                imgs.append(torch.zeros(3, 224, 224))

        batch = torch.stack(imgs).to(DEVICE, non_blocking=True)

        feat = model.encode_image(batch).float()
        feat = feat / (feat.norm(dim=-1, keepdim=True) + 1e-8)

        embs.append(feat.cpu().numpy())

    return np.vstack(embs)

In [8]:
# ── Dataset for contrastive fine-tuning ───────────────────────────────────────
class FashionPairDataset(Dataset):
    """Sample positive pairs (two crops of same item_id) for InfoNCE."""
    def __init__(self, df: pd.DataFrame, preprocess, seed: int = 42):
        np.random.seed(seed)
        groups = df.groupby('item_id')
        self.valid_ids  = [k for k, v in groups if len(v) >= 2]
        self.groups     = groups
        self.preprocess = preprocess

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        iid    = self.valid_ids[idx]
        group  = self.groups.get_group(iid)
        sample = group.sample(2)
        imgs   = []
        for _, row in sample.iterrows():
            try:
                img = Image.open(crop_path(row)).convert('RGB')
            except:
                img = Image.fromarray(np.zeros((224,224,3), dtype=np.uint8))
            imgs.append(self.preprocess(img))
        return imgs[0], imgs[1]

# ── CLIP fine-tuning function ─────────────────────────────────────────────────
def finetune_clip(seed: int, train_df: pd.DataFrame) -> tuple:
    """
    Fine-tune CLIP visual encoder (last 4 transformer blocks min; full here)
    using InfoNCE contrastive loss on positive pairs.
    Returns: (fine-tuned clip model, preprocess)
    """
    set_seed(seed)

    model, preprocess = clip.load(CLIP_MODEL, device=DEVICE, jit=False)
    model = model.float()

    # Fine-tune full visual encoder (spec allows this; last 4 blocks is minimum)
    for p in model.parameters():
        p.requires_grad_(False)
    for p in model.visual.parameters():
        p.requires_grad_(True)
    # Text encoder stays frozen per spec

    dataset = FashionPairDataset(train_df, preprocess, seed=seed)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True,
                         persistent_workers=True)

    optimizer = torch.optim.AdamW(model.visual.parameters(), lr=FINETUNE_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=FINETUNE_EPOCHS * len(loader))
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler()

    model.train()
    pbar = tqdm(total=FINETUNE_EPOCHS * len(loader),
                desc=f'Fine-tune seed={seed}', unit='step')

    for epoch in range(FINETUNE_EPOCHS):
        epoch_loss = 0.0
        for img1, img2 in loader:
            img1 = img1.to(DEVICE, non_blocking=True)
            img2 = img2.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                f1 = model.encode_image(img1).float()
                f2 = model.encode_image(img2).float()
                f1 = f1 / (f1.norm(dim=-1, keepdim=True) + 1e-8)
                f2 = f2 / (f2.norm(dim=-1, keepdim=True) + 1e-8)

                logits = (f1 @ f2.T) / TEMPERATURE
                logits = logits.clamp(-50, 50)
                labels = torch.arange(logits.shape[0], device=DEVICE)
                loss   = (criterion(logits, labels) + criterion(logits.T, labels)) / 2

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.visual.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            epoch_loss += loss.item()
            pbar.update(1)
            pbar.set_postfix(epoch=epoch+1, loss=f'{loss.item():.4f}')

        print(f'  Epoch {epoch+1}/{FINETUNE_EPOCHS} | avg_loss={epoch_loss/len(loader):.4f}')

    pbar.close()
    model.eval()
    return model, preprocess

In [14]:
gallery_ids = gallery_df['item_id'].values
query_ids   = query_df['item_id'].values

In [16]:
print('=' * 60)
print('EXPERIMENT C — Fine-tuned CLIP + Frozen BLIP-2')
print(f'Alpha values tested: {ALPHA_C}')
print(f'Seeds: {SEEDS}')
print('=' * 60)

records_C  = []   # one dict per (seed, alpha)
loss_curves = {}  # seed -> list of epoch avg losses (for viz)

pbar_C_outer = tqdm(SEEDS, desc='Exp C Seeds', unit='seed')

for seed in pbar_C_outer:
    pbar_C_outer.set_postfix(seed=seed)
    print(f'\n── Seed {seed} ──────────────────────────────────────────')

    # ── Fine-tune CLIP ────────────────────────────────────────────────────────
    ft_model, ft_preprocess = finetune_clip(seed, train_df)

    # ── Generate fine-tuned image embeddings ─────────────────────────────────
    gallery_img_emb_ft = encode_images_clip(gallery_df, ft_model, ft_preprocess)
    query_img_emb_ft   = encode_images_clip(query_df,   ft_model, ft_preprocess)

    # Save per-seed embeddings (useful for online pipeline / demo)
    np.save(EMB_DIR / f'gallery_img_ft_seed{seed}.npy', gallery_img_emb_ft)
    np.save(EMB_DIR / f'query_img_ft_seed{seed}.npy',   query_img_emb_ft)

    # Save model weights (last seed will be 'best' model for demo)
    torch.save(ft_model.visual.state_dict(),
               WORK / f'clip_visual_ft_seed{seed}.pth')

    # ── Evaluate at each alpha ────────────────────────────────────────────────
    for alpha in ALPHA_C:
        g_fused = fuse(gallery_img_emb_ft.copy(), gallery_txt_emb.copy(), alpha)
        q_fused = fuse(query_img_emb_ft.copy(),   query_txt_emb.copy(),   alpha)

        topk = build_hnsw_and_retrieve(g_fused, q_fused)
        m    = compute_all_metrics(topk, query_ids, gallery_ids)
        m['seed']  = seed
        m['alpha'] = alpha
        records_C.append(m)
        print(f'  α={alpha} | ', '  '.join([f'{k}: {v:.4f}' for k,v in m.items() if '@' in k]))

    # Cleanup fine-tuned model from VRAM
    del ft_model; gc.collect(); torch.cuda.empty_cache()

# ── Build best model HNSW index for online pipeline ──────────────────────────
# Reload best seed's embeddings (last seed, best alpha)
df_C_full = pd.DataFrame(records_C)
best_seed_C  = df_C_full.groupby('seed')['mAP@10'].mean().idxmax()
best_alpha_C = ALPHA_C[np.argmax([df_C_full[df_C_full['alpha']==a]['mAP@10'].mean() for a in ALPHA_C])]

print(f'\nBest seed: {best_seed_C}, Best alpha: {best_alpha_C}')

gallery_img_emb_best = np.load(EMB_DIR / f'gallery_img_ft_seed{best_seed_C}.npy')
query_img_emb_best   = np.load(EMB_DIR / f'query_img_ft_seed{best_seed_C}.npy')

g_fused_C = fuse(gallery_img_emb_best.copy(), gallery_txt_emb.copy(), best_alpha_C)
q_fused_C = fuse(query_img_emb_best.copy(),   query_txt_emb.copy(),   best_alpha_C)

topk_C_best = build_hnsw_and_retrieve(g_fused_C, q_fused_C)
np.save(IDX_DIR / 'topk_C_best.npy', topk_C_best)

# Save HNSW index for online pipeline
dim_C   = g_fused_C.shape[1]
idx_C   = faiss.IndexHNSWFlat(dim_C, 32)
idx_C.hnsw.efConstruction = 200
idx_C.hnsw.efSearch       = 128
faiss.normalize_L2(g_fused_C)
idx_C.add(g_fused_C)
faiss.write_index(idx_C, str(IDX_DIR / 'fashion_hnsw_best.index'))
np.save(IDX_DIR / 'gallery_fused_best.npy', g_fused_C)
print('HNSW index saved for online pipeline.')

# Save results
df_C_full.to_csv(RESULT_DIR / 'exp_C_all_seeds.csv', index=False)

# ── Summary: mean ± std per alpha ─────────────────────────────────────────────
print('\n── Experiment C: Mean ± Std (across seeds) per Alpha ──')
summary_C_rows = []
for alpha in ALPHA_C:
    sub = df_C_full[df_C_full['alpha'] == alpha]
    row = {'alpha': alpha}
    for col in [c for c in df_C_full.columns if '@' in c]:
        row[col] = f"{sub[col].mean():.4f} ± {sub[col].std(ddof=1):.4f}"
    summary_C_rows.append(row)

df_C_summary = pd.DataFrame(summary_C_rows)
df_C_summary.to_csv(RESULT_DIR / 'exp_C_summary.csv', index=False)
print(df_C_summary.to_string(index=False))

# ── Plot: metric vs K for each alpha ─────────────────────────────────────────


EXPERIMENT C — Fine-tuned CLIP + Frozen BLIP-2
Alpha values tested: [0.3, 0.7]
Seeds: [42, 137, 256, 512]


Exp C Seeds:   0%|          | 0/4 [00:00<?, ?seed/s]


── Seed 42 ──────────────────────────────────────────


Fine-tune seed=42:   0%|          | 0/620 [00:00<?, ?step/s]

  Epoch 1/5 | avg_loss=1.3646
  Epoch 2/5 | avg_loss=0.6889
  Epoch 3/5 | avg_loss=0.5768
  Epoch 4/5 | avg_loss=0.5172
  Epoch 5/5 | avg_loss=0.4904


...

  Epoch 1/5 | avg_loss=1.3864
  Epoch 2/5 | avg_loss=0.7119
  Epoch 3/5 | avg_loss=0.5604
  Epoch 4/5 | avg_loss=0.5186
  Epoch 5/5 | avg_loss=0.5031


CLIP Image Emb:   0%|          | 0/99 [00:00<?, ?batch/s]

CLIP Image Emb:   0%|          | 0/112 [00:00<?, ?batch/s]

  α=0.3 |  Recall@5: 0.4247  mAP@5: 0.3111  NDCG@5: 0.3423  Recall@10: 0.5106  mAP@10: 0.3100  NDCG@10: 0.3650  Recall@15: 0.5639  mAP@15: 0.3048  NDCG@15: 0.3752
  α=0.7 |  Recall@5: 0.7597  mAP@5: 0.6055  NDCG@5: 0.6524  Recall@10: 0.8296  mAP@10: 0.5845  NDCG@10: 0.6623  Recall@15: 0.8628  mAP@15: 0.5650  NDCG@15: 0.6616

Best seed: 137, Best alpha: 0.7
HNSW index saved for online pipeline.

── Experiment C: Mean ± Std (across seeds) per Alpha ──
 alpha        Recall@5           mAP@5          NDCG@5       Recall@10          mAP@10         NDCG@10       Recall@15          mAP@15         NDCG@15
   0.3 0.4240 ± 0.0006 0.3119 ± 0.0008 0.3426 ± 0.0006 0.5113 ± 0.0014 0.3109 ± 0.0008 0.3657 ± 0.0009 0.5642 ± 0.0010 0.3052 ± 0.0006 0.3757 ± 0.0007
   0.7 0.7620 ± 0.0021 0.6055 ± 0.0013 0.6529 ± 0.0012 0.8301 ± 0.0007 0.5838 ± 0.0013 0.6621 ± 0.0011 0.8630 ± 0.0013 0.5644 ± 0.0013 0.6613 ± 0.0007


NameError: name 'colors' is not defined

In [18]:
colors = ['steelblue', 'darkorange']
sample_q_idx = [0, 10, 25, 50, 100]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ['Recall', 'mAP', 'NDCG']):
    for alpha, color in zip(ALPHA_C, colors):
        sub   = df_C_full[df_C_full['alpha'] == alpha]
        means = [sub[f'{metric}@{k}'].mean() for k in K_VALUES]
        stds  = [sub[f'{metric}@{k}'].std(ddof=1) for k in K_VALUES]
        ax.plot(K_VALUES, means, marker='o', color=color, label=f'α={alpha}')
        ax.fill_between(K_VALUES,
                        [m-s for m,s in zip(means,stds)],
                        [m+s for m,s in zip(means,stds)],
                        alpha=0.2, color=color)
    ax.set_title(f'Exp C — {metric}@K')
    ax.set_xlabel('K'); ax.set_ylabel(metric)
    ax.set_ylim(0, 1); ax.grid(True); ax.legend()
plt.suptitle('Experiment C — Fine-tuned CLIP + Frozen BLIP-2 (mean ± std)', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_C_metrics.png', dpi=150)
plt.close()
print('Exp C metrics plot saved.')

# Retrieval visualization for best config
fig, axes = plt.subplots(5, MAX_K+1, figsize=(22, 18))
for row_i, qi in enumerate(sample_q_idx):
    qrow = query_df.iloc[qi]
    axes[row_i, 0].imshow(Image.open(crop_path(qrow)).convert('RGB'))
    axes[row_i, 0].set_title('Query', fontsize=8); axes[row_i, 0].axis('off')
    for col_i, gi in enumerate(topk_C_best[qi, :MAX_K]):
        grow  = gallery_df.iloc[gi]
        match = grow['item_id'] == qrow['item_id']
        axes[row_i, col_i+1].imshow(Image.open(crop_path(grow)).convert('RGB'))
        axes[row_i, col_i+1].set_title('✓' if match else '✗',
                                        color='green' if match else 'red', fontsize=9)
        axes[row_i, col_i+1].axis('off')
plt.suptitle(f'Exp C — Top-15 Retrieval (α={best_alpha_C}, seed={best_seed_C})', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'exp_C_retrieval.png', dpi=100)
plt.close()
print('Exp C retrieval visualization saved.')

Exp C metrics plot saved.
Exp C retrieval visualization saved.


## 10. Step 4 — BLIP-2 ITM Re-ranking (Best Model, Exp C)

In [20]:
gallery_cap_df = pd.read_csv(
    '/kaggle/working/captions/gallery_captions.csv'
)

query_cap_df = pd.read_csv(
    '/kaggle/working/captions/query_captions.csv'
)

In [24]:
print('Loading BLIP ITM model for re-ranking...')
itm_processor = BlipProcessor.from_pretrained('Salesforce/blip-itm-base-coco')
itm_model = BlipForImageTextRetrieval.from_pretrained(
    'Salesforce/blip-itm-base-coco',
    torch_dtype=torch.float16
).to(DEVICE).eval()

@torch.no_grad()
def itm_score_batch(query_img: Image.Image, captions: list, sub_batch: int = 32) -> np.ndarray:
    """
    Use use_itm_head=False: computes cosine similarity between
    query image embedding and each gallery caption embedding.
    More robust than ITM binary head for cross-view fashion retrieval.
    """
    img_inputs   = itm_processor(images=query_img, return_tensors='pt')
    pixel_values = img_inputs['pixel_values'].to(DEVICE, dtype=torch.float16)

    vision_out   = itm_model.vision_model(pixel_values=pixel_values)
    image_embeds = vision_out.last_hidden_state                          # (1, seq, dim)
    image_feat   = torch.nn.functional.normalize(
        itm_model.vision_proj(image_embeds[:, 0, :]), dim=-1
    )                                                                    # (1, proj_dim)

    all_scores = []
    for i in range(0, len(captions), sub_batch):
        batch_caps = captions[i:i+sub_batch]
        B = len(batch_caps)

        txt = itm_processor(
            text=batch_caps,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=64
        )
        input_ids      = txt['input_ids'].to(DEVICE)
        attention_mask = txt['attention_mask'].to(DEVICE)

        q_out = itm_model.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        text_feat = torch.nn.functional.normalize(
            itm_model.text_proj(q_out.last_hidden_state[:, 0, :]), dim=-1
        )                                                                # (B, proj_dim)

        # cosine similarity: (1, proj_dim) x (proj_dim, B) → (B,)
        scores = (image_feat @ text_feat.T).squeeze(0)                  # (B,)
        all_scores.append(scores.float().cpu().numpy())

    return np.concatenate(all_scores)


print('Running ITM re-ranking on best Exp C results...')
topk_before  = topk_C_best
all_captions = gallery_cap_df['caption'].fillna('clothing item').tolist()
reranked     = np.empty_like(topk_before)

pbar_itm = tqdm(range(len(query_df)), desc='ITM Re-ranking', unit='query')
for qi in pbar_itm:
    qrow     = query_df.iloc[qi]
    q_img    = Image.open(crop_path(qrow)).convert('RGB')
    cand_idx = topk_before[qi]                        # (MAX_K,)
    caps     = [all_captions[gi] for gi in cand_idx]
    scores   = itm_score_batch(q_img, caps, sub_batch=MAX_K)   # all 15 in one shot
    order    = np.argsort(-scores)
    reranked[qi] = cand_idx[order]

pbar_itm.close()

np.save(IDX_DIR / 'topk_C_itm_reranked.npy', reranked)

metrics_before   = compute_all_metrics(topk_before, query_ids, gallery_ids)
metrics_reranked = compute_all_metrics(reranked,    query_ids, gallery_ids)

cmp_df = pd.DataFrame([
    {'Stage': 'Before ITM', **metrics_before},
    {'Stage': 'After ITM',  **metrics_reranked}
])
cmp_df.to_csv(RESULT_DIR / 'itm_reranking_comparison.csv', index=False)
print('\n── ITM Re-ranking Effect ──')
print(cmp_df.set_index('Stage').T.to_string())

del itm_model; gc.collect(); torch.cuda.empty_cache()

Loading BLIP ITM model for re-ranking...


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running ITM re-ranking on best Exp C results...


ITM Re-ranking:   0%|          | 0/14218 [00:00<?, ?query/s]


── ITM Re-ranking Effect ──
Stage      Before ITM  After ITM
Recall@5     0.763047   0.668519
mAP@5        0.607335   0.426795
NDCG@5       0.654531   0.493557
Recall@10    0.830919   0.814109
mAP@10       0.585250   0.421017
NDCG@10      0.663484   0.533514
Recall@15    0.861162   0.861162
mAP@15       0.565811   0.408955
NDCG@15      0.662081   0.541433


## 11. Ablation Comparison — A vs B vs C

In [32]:
df_A_saved = pd.read_csv(
    RESULT_DIR / 'exp_A_metrics.csv'
)

metrics_A = df_A_saved.iloc[0].to_dict()

df_B_full = pd.read_csv(
    RESULT_DIR / 'exp_B_all_seeds.csv'
)

df_C_full = pd.read_csv(
    RESULT_DIR / 'exp_C_all_seeds.csv'
)

best_alpha_B = ALPHA_B[
    np.argmax([
        df_B_full[df_B_full['alpha'] == a]['mAP@10'].mean()
        for a in ALPHA_B
    ])
]

best_alpha_C = ALPHA_C[
    np.argmax([
        df_C_full[df_C_full['alpha'] == a]['mAP@10'].mean()
        for a in ALPHA_C
    ])
]

topk_A = np.load(
    '/kaggle/working/indices/topk_A.npy'
)

topk_B_best = np.load(
    '/kaggle/working/indices/topk_B_best.npy'
)

topk_C_best = np.load(
    '/kaggle/working/indices/topk_C_best.npy'
)

reranked = np.load(
    '/kaggle/working/indices/topk_C_itm_reranked.npy'
)

In [33]:
print('=' * 60)
print('ABLATION COMPARISON: A vs B vs C')
print('=' * 60)

# Aggregate representative numbers
row_A = {'Experiment': 'A: Vision-only CLIP (α=1)', **metrics_A}

# Best alpha for B and C
sub_B_best = df_B_full[df_B_full['alpha'] == best_alpha_B]
sub_C_best = df_C_full[df_C_full['alpha'] == best_alpha_C]

row_B = {'Experiment': f'B: Frozen CLIP+BLIP-2 (α={best_alpha_B})'}
row_C = {'Experiment': f'C: Fine-tuned CLIP+BLIP-2 (α={best_alpha_C})'}
row_C_itm = {'Experiment': f'C+ITM: Fine-tuned+Reranked (α={best_alpha_C})'}

for col in [c for c in df_B_full.columns if '@' in c]:
    row_B[col]     = f"{sub_B_best[col].mean():.4f} ± {sub_B_best[col].std(ddof=1):.4f}"
    row_C[col]     = f"{sub_C_best[col].mean():.4f} ± {sub_C_best[col].std(ddof=1):.4f}"
    row_C_itm[col] = f"{metrics_reranked[col]:.4f}"
    row_A[col]     = f"{metrics_A[col]:.4f}"

comparison_df = pd.DataFrame([row_A, row_B, row_C, row_C_itm])
comparison_df.to_csv(RESULT_DIR / 'ablation_comparison.csv', index=False)

print(comparison_df.to_string(index=False))

# ── Plot: grouped bar chart for K=10 ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
exp_labels = ['A\nVision-only', f'B\nFrozen\nα={best_alpha_B}',
              f'C\nFine-tuned\nα={best_alpha_C}', 'C+ITM']

for ax, metric in zip(axes, ['Recall', 'mAP', 'NDCG']):
    vals = [
        metrics_A[f'{metric}@10'],
        sub_B_best[f'{metric}@10'].mean(),
        sub_C_best[f'{metric}@10'].mean(),
        metrics_reranked[f'{metric}@10']
    ]
    errs = [
        0,
        sub_B_best[f'{metric}@10'].std(ddof=1),
        sub_C_best[f'{metric}@10'].std(ddof=1),
        0
    ]
    bars = ax.bar(exp_labels, vals, yerr=errs, capsize=5,
                  color=['steelblue','darkorange','green','red'], alpha=0.8)
    ax.set_title(f'{metric}@10')
    ax.set_ylim(0, 1); ax.set_ylabel(metric); ax.grid(axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Ablation Study — A vs B vs C vs C+ITM at K=10', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'ablation_comparison.png', dpi=150)
plt.close()

# ── Full metric table across all K values ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
styles = [('A', 'steelblue', '-', 'o'),
          ('B', 'darkorange', '--', 's'),
          ('C', 'green', '-.', '^'),
          ('C+ITM', 'red', ':', 'D')]
topk_list = [topk_A, topk_B_best, topk_C_best, reranked]

for ax, metric in zip(axes, ['Recall', 'mAP', 'NDCG']):
    for (label, color, ls, marker), topk in zip(styles, topk_list):
        vals = [getattr(__import__('__main__'), f'{metric.lower()}_at_k')(topk, query_ids, gallery_ids, k)
                for k in K_VALUES]
        ax.plot(K_VALUES, vals, color=color, ls=ls, marker=marker,
                linewidth=2, label=label)
    ax.set_title(f'{metric}@K — All Experiments')
    ax.set_xlabel('K'); ax.set_ylabel(metric)
    ax.set_ylim(0, 1); ax.grid(True); ax.legend()
plt.suptitle('Ablation Study — All Experiments across K', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'ablation_all_k.png', dpi=150)
plt.close()

print('\nAll comparison plots saved.')

ABLATION COMPARISON: A vs B vs C
                        Experiment        Recall@5           mAP@5          NDCG@5       Recall@10          mAP@10         NDCG@10       Recall@15          mAP@15         NDCG@15
         A: Vision-only CLIP (α=1)          0.3602          0.2568          0.2850          0.4334          0.2558          0.3043          0.4768          0.2519          0.3129
     B: Frozen CLIP+BLIP-2 (α=0.7) 0.4004 ± 0.0000 0.2774 ± 0.0000 0.3111 ± 0.0000 0.4908 ± 0.0000 0.2775 ± 0.0000 0.3355 ± 0.0000 0.5436 ± 0.0000 0.2721 ± 0.0000 0.3457 ± 0.0000
 C: Fine-tuned CLIP+BLIP-2 (α=0.7) 0.7620 ± 0.0021 0.6055 ± 0.0013 0.6529 ± 0.0012 0.8301 ± 0.0007 0.5838 ± 0.0013 0.6621 ± 0.0011 0.8630 ± 0.0013 0.5644 ± 0.0013 0.6613 ± 0.0007
C+ITM: Fine-tuned+Reranked (α=0.7)          0.6685          0.4268          0.4936          0.8141          0.4210          0.5335          0.8612          0.4090          0.5414

All comparison plots saved.


## 12. Final Summary & Saved Artifacts

In [34]:
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)

print('\n── Experiment A (Baseline) ──')
for k in K_VALUES:
    print(f'  K={k:2d}  Recall={metrics_A[f"Recall@{k}"]:.4f}  '
          f'mAP={metrics_A[f"mAP@{k}"]:.4f}  '
          f'NDCG={metrics_A[f"NDCG@{k}"]:.4f}')

print(f'\n── Experiment B (best α={best_alpha_B}) ──')
for k in K_VALUES:
    col = f'Recall@{k}'; sub = sub_B_best
    print(f'  K={k:2d}  Recall={sub[f"Recall@{k}"].mean():.4f}±{sub[f"Recall@{k}"].std(ddof=1):.4f}  '
          f'mAP={sub[f"mAP@{k}"].mean():.4f}±{sub[f"mAP@{k}"].std(ddof=1):.4f}  '
          f'NDCG={sub[f"NDCG@{k}"].mean():.4f}±{sub[f"NDCG@{k}"].std(ddof=1):.4f}')

print(f'\n── Experiment C (best α={best_alpha_C}) ──')
for k in K_VALUES:
    sub = sub_C_best
    print(f'  K={k:2d}  Recall={sub[f"Recall@{k}"].mean():.4f}±{sub[f"Recall@{k}"].std(ddof=1):.4f}  '
          f'mAP={sub[f"mAP@{k}"].mean():.4f}±{sub[f"mAP@{k}"].std(ddof=1):.4f}  '
          f'NDCG={sub[f"NDCG@{k}"].mean():.4f}±{sub[f"NDCG@{k}"].std(ddof=1):.4f}')

print(f'\n── Experiment C + ITM Re-ranking ──')
for k in K_VALUES:
    print(f'  K={k:2d}  Recall={metrics_reranked[f"Recall@{k}"]:.4f}  '
          f'mAP={metrics_reranked[f"mAP@{k}"]:.4f}  '
          f'NDCG={metrics_reranked[f"NDCG@{k}"]:.4f}')

print('\n── Saved Artifacts ──')
print(f'  Crops          : {CROP_DIR}')
print(f'  Embeddings     : {EMB_DIR}')
print(f'  Captions       : {CAP_DIR}')
print(f'  HNSW Index     : {IDX_DIR}/fashion_hnsw_best.index')
print(f'  Gallery fused  : {IDX_DIR}/gallery_fused_best.npy')
print(f'  Model weights  : {WORK}/clip_visual_ft_seed*.pth')
print(f'  Results CSVs   : {RESULT_DIR}')
print(f'  Visualizations : {VIZ_DIR}')
print('\nAll done!')

FINAL SUMMARY

── Experiment A (Baseline) ──
  K= 5  Recall=0.3602  mAP=0.2568  NDCG=0.2850
  K=10  Recall=0.4334  mAP=0.2558  NDCG=0.3043
  K=15  Recall=0.4768  mAP=0.2519  NDCG=0.3129

── Experiment B (best α=0.7) ──
  K= 5  Recall=0.4004±0.0000  mAP=0.2774±0.0000  NDCG=0.3111±0.0000
  K=10  Recall=0.4908±0.0000  mAP=0.2775±0.0000  NDCG=0.3355±0.0000
  K=15  Recall=0.5436±0.0000  mAP=0.2721±0.0000  NDCG=0.3457±0.0000

── Experiment C (best α=0.7) ──
  K= 5  Recall=0.7620±0.0021  mAP=0.6055±0.0013  NDCG=0.6529±0.0012
  K=10  Recall=0.8301±0.0007  mAP=0.5838±0.0013  NDCG=0.6621±0.0011
  K=15  Recall=0.8630±0.0013  mAP=0.5644±0.0013  NDCG=0.6613±0.0007

── Experiment C + ITM Re-ranking ──
  K= 5  Recall=0.6685  mAP=0.4268  NDCG=0.4936
  K=10  Recall=0.8141  mAP=0.4210  NDCG=0.5335
  K=15  Recall=0.8612  mAP=0.4090  NDCG=0.5414

── Saved Artifacts ──
  Crops          : /kaggle/working/cropped
  Embeddings     : /kaggle/working/embeddings
  Captions       : /kaggle/working/captions
  HNSW